# 03 — TopK / Spread Research

**目标：** 手动验证 `select_topk` / `select_bottomk` / `build_rolling_portfolio` / `compute_spread` 的完整链路。

前置条件：已完成 `02_signal_validation.ipynb` 并确认 IC 正常。

In [ ]:
import sys, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ── Core interfaces ──────────────────────────────────────────
from src.core.signals import generate_scores_panel
from src.core.selection import select_topk, select_bottomk, select_topk_bottomk
from src.core.portfolio import build_rolling_portfolio
from src.core.metrics import compute_spread, compute_ic_series
from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init

print('✅ src.core interfaces imported')

## 0. 配置

In [ ]:
MARKET       = 'us'
MODEL_TAG    = 'us_absret'
TEST_START   = '2025-01-01'
TEST_END     = '2026-06-18'
SYMBOLS      = ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','COST','NFLX']
BENCHMARK    = 'QQQ'
K            = 5   # 每腿股票数
HOLDING_DAYS = 10  # 换仓间隔（交易日）

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D
from qlib.contrib.data.loader import Alpha158DL
print(f'Config: K={K}  holding_days={HOLDING_DAYS}  bench={BENCHMARK}')

## 1. 加载模型 + 数据

In [ ]:
import time
from src.assistant.model_registry_index import ModelRegistryIndex
from src.assistant.metadata_db import resolve_metadata_db_path

# 加载模型
reg = ModelRegistryIndex(db_path=resolve_metadata_db_path(ROOT))
entries = [e for e in reg.list_entries() if e.get('tag') == MODEL_TAG]
assert entries, f'Run end_to_end_training_pipeline first (tag={MODEL_TAG})'
latest = sorted(entries, key=lambda e: e.get('created_at',''))[-1]
with open(latest['path'], 'rb') as f:
    booster = pickle.load(f)
print(f'Model: {latest["tag"]}  registered={latest["created_at"]}')

# 加载特征 + Label
alpha_exprs, _ = Alpha158DL.get_feature_config({'kbar':{},'price':{'windows':[0],'feature':['OPEN','HIGH','LOW','VWAP']},'rolling':{}})
extras = ['$close/Ref($close,5)-1','$close/Ref($close,10)-1','$close/Ref($close,20)-1','Std($close,10)','$volume/Ref($volume,10)-1']
t0 = time.perf_counter()
X_test = D.features(SYMBOLS, list(alpha_exprs)+extras, start_time=TEST_START, end_time=TEST_END)
X_test = X_test.fillna(0.0).replace([float('inf'),float('-inf')], 0.0)
y_raw  = D.features(SYMBOLS, ['Ref($close,-10)/Ref($close,-1)-1'], start_time=TEST_START, end_time=TEST_END)
return_panel = y_raw.rename(columns={y_raw.columns[0]:'return'})
print(f'Data loaded in {time.perf_counter()-t0:.1f}s  X={X_test.shape}')

# 基准收益
try:
    bench_raw = D.features([BENCHMARK], ['Ref($close,-10)/Ref($close,-1)-1'],
                           start_time=TEST_START, end_time=TEST_END)
    bench_returns = bench_raw.xs(BENCHMARK, level='instrument').iloc[:,0]
except Exception:
    bench_returns = None
    print('⚠️ Benchmark data unavailable')

## 2. 单截面验证 — `select_topk` / `select_bottomk`

In [ ]:
score_panel = generate_scores_panel(booster, X_test)

dates = sorted(score_panel.index.get_level_values(0 if score_panel.index.names[0]=='datetime' else 1).unique())
latest_date = dates[-1]

if score_panel.index.names[0] == 'datetime':
    scores_today = score_panel.xs(latest_date, level=0)['score']
else:
    scores_today = score_panel.xs(latest_date, level='datetime')['score']

# ── 调用 src.core 接口 ─────────────────────────────────────
top_k = select_topk(scores_today, k=K, guardrail=False)   # 先不加 guardrail 做基础验证
bot_k = select_bottomk(scores_today, k=K)
portfolio = select_topk_bottomk(scores_today, k_long=K, k_short=K)

print(f'Date: {latest_date}')
print(f'LONG  (Top-{K}): {top_k}')
print(f'SHORT (Bot-{K}): {bot_k}')
print()
# 分数差距
print(f'Long avg score:  {scores_today[top_k].mean():.4f}')
print(f'Short avg score: {scores_today[bot_k].mean():.4f}')
print(f'Score spread:    {scores_today[top_k].mean() - scores_today[bot_k].mean():.4f}')

## 3. 滚动回测 — `build_rolling_portfolio()`

In [ ]:
# ── 调用 src.core 接口 ─────────────────────────────────────
result = build_rolling_portfolio(
    score_panel=score_panel,
    return_panel=return_panel,
    k=K,
    holding_days=HOLDING_DAYS,
    long_only=False,
    guardrail=False,   # 先验证无 guardrail 基线
    benchmark_returns=bench_returns,
)

print('=== Rolling Portfolio (no guardrail) ===')
print(f'  Long leg days:  {len(result.long_returns)}')
print(f'  Rebalance dates: {len(result.long_holdings)}')
print(f'  Sample long holding ({list(result.long_holdings.keys())[0]}): {list(result.long_holdings.values())[0]}')

## 4. Spread 分析 — `compute_spread()`

In [ ]:
# ── 调用 src.core 接口 ─────────────────────────────────────
spread_stats = compute_spread(
    long_returns=result.long_returns,
    short_returns=result.short_returns,
    bench_returns=result.bench_returns if len(result.bench_returns)>0 else None,
)

print('=== Spread Statistics ===')
print(f'  Spread Mean:   {spread_stats["spread_mean"]:.4f}')
print(f'  Spread Std:    {spread_stats["spread_std"]:.4f}')
print(f'  Spread Sharpe: {spread_stats["spread_sharpe"]:.2f}')
print(f'  Alpha Long:    {spread_stats["alpha_long"]:.4f}')
print(f'  Alpha Short:   {spread_stats["alpha_short"]:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Equity curves
result.long_equity.plot(ax=axes[0], color='#22c55e', lw=1.5, label=f'Long Top-{K}')
(1+result.short_returns).cumprod().plot(ax=axes[0], color='#ef4444', lw=1.5, label=f'Short Bot-{K}')
if len(result.bench_returns) > 0:
    (1+result.bench_returns).cumprod().plot(ax=axes[0], color='#f59e0b', lw=1, ls='--', label=BENCHMARK)
axes[0].axhline(1, color='gray', ls=':', lw=0.8); axes[0].legend(fontsize=8)
axes[0].set_title(f'Equity Curves (K={K})')

# Spread equity
spread_stats['spread_equity'].plot(ax=axes[1], color='#6366f1', lw=1.5)
axes[1].axhline(1, color='gray', ls=':', lw=0.8)
axes[1].set_title(f'Long-Short Spread Equity (Sharpe={spread_stats["spread_sharpe"]:.2f})')

# Daily spread distribution
axes[2].hist(spread_stats['spread_series'], bins=40, color='#8b5cf6', alpha=0.8, edgecolor='white')
axes[2].axvline(0, color='red', ls='--', lw=1)
axes[2].axvline(spread_stats['spread_mean'], color='green', ls='--', lw=1)
axes[2].set_title('Daily Spread Distribution')
plt.tight_layout(); plt.show()

## 5. 参数敏感性 — K 和 holding_days 扫描

In [ ]:
# ── 你可以直接修改参数，无需改任何核心代码 ────────────────
print('K scan (holding_days fixed at 10):')
for k in [3, 5, 8, 10]:
    r = build_rolling_portfolio(score_panel, return_panel, k=k, holding_days=10, guardrail=False)
    s = compute_spread(r.long_returns, r.short_returns)
    print(f'  K={k:2d}: spread_mean={s["spread_mean"]:.4f}  sharpe={s["spread_sharpe"]:.2f}')

print()
print('holding_days scan (K fixed at 5):')
for hd in [5, 10, 15, 20]:
    r = build_rolling_portfolio(score_panel, return_panel, k=5, holding_days=hd, guardrail=False)
    s = compute_spread(r.long_returns, r.short_returns)
    print(f'  holding={hd:2d}d: spread_mean={s["spread_mean"]:.4f}  sharpe={s["spread_sharpe"]:.2f}')

## 6. Guardrail 对比

In [ ]:
# 无 guardrail
r_no_guard = build_rolling_portfolio(score_panel, return_panel, k=K, holding_days=HOLDING_DAYS, guardrail=False)
s_no = compute_spread(r_no_guard.long_returns, r_no_guard.short_returns)

# 有 guardrail（score>0 threshold only，无 price/MA 数据时 select_topk 自动降级）
r_guard = build_rolling_portfolio(score_panel, return_panel, k=K, holding_days=HOLDING_DAYS, guardrail=True)
s_gd = compute_spread(r_guard.long_returns, r_guard.short_returns)

print('Guardrail comparison:')
print(f'  No guardrail: spread_sharpe={s_no["spread_sharpe"]:.2f}  mean={s_no["spread_mean"]:.4f}')
print(f'  Guardrail:    spread_sharpe={s_gd["spread_sharpe"]:.2f}  mean={s_gd["spread_mean"]:.4f}')

fig, ax = plt.subplots(figsize=(12, 4))
s_no['spread_equity'].plot(ax=ax, color='#ef4444', lw=1.5, label='No guardrail')
s_gd['spread_equity'].plot(ax=ax, color='#22c55e', lw=1.5, label='With guardrail')
ax.axhline(1, color='gray', ls=':', lw=0.8)
ax.legend(fontsize=9); ax.set_title(f'Guardrail Effect (K={K})')
plt.tight_layout(); plt.show()

## 7. 结论

- [ ] Spread Sharpe > 0 — 信号有方向性
- [ ] Top leg > Benchmark — long leg 有效
- [ ] Bottom leg < Benchmark — short leg 有效
- [ ] Guardrail 是否提升？记录最优参数
- [ ] 最优 K = ___  holding_days = ___

确认以上后，将最优参数更新到 `end_to_end_training_pipeline.ipynb` Step 0 配置。